## Capstone Project of RML (Group 4) - HMDA dataset 

### Pre-modeling 

#### 1.	Data inspection

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

# Load local dataset
file_path = "2024_lar.txt.csv"
raw_data = pd.read_csv(file_path, sep=",")

# Number of rows
raw_data.shape[0]

In [ ]:
raw_data.head(10)

In [ ]:
list(raw_data.columns)

In [ ]:
raw_data.info()

#### 2.	Target construction

We build the target from `action_taken`. Codes 1 and 2 (originated / approved but not taken) = 1, code 3 (denied) = 0. We drop the rest (withdrawn, incomplete, purchased, etc.) because they are not a clear approve/deny decision.

In [ ]:
import numpy as np

# Create a copy of the original dataset to avoid modifying raw data
df = raw_data.copy()

# Step 1: Filter the dataset to keep only relevant action_taken values
# Keep:
# 1 = Loan originated
# 2 = Approved but not accepted
# 3 = Denied
# Remove all other categories (e.g., withdrawn, incomplete, etc.)
df = df[df["action_taken"].isin([1, 2, 3])]

# Step 2: Create a binary target variable
# Approved (1, 2) → 1
# Denied (3) → 0
df["target"] = np.where(
    df["action_taken"].isin([1, 2]),  # approved cases
    1,
    0  # denied cases
)

# Step 3: Check the distribution of the target variable
# This helps us understand class balance (approved vs denied)
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True))


Roughly three quarters of rows are approves and one quarter are denies, so accuracy alone is misleading — a constant "approve" model already looks good. We look at FPR and FNR by group instead of stopping at overall accuracy.

#### 3. Data cleaning

**Converting values into `NaN`**

HMDA encodes missings as the strings `"NA"` and `"Exempt"`, so pandas would treat them as real categories. We swap those to `NaN` before imputation and one-hot encoding.

In [ ]:
df = df.replace(["NA", "Exempt"], np.nan)

**Identifying Missing Values**

We rank columns by % missing to see which are basically empty; those are not useful to keep as-is.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(20)

**Dropping Columns with More Than 90% Missing Values**

If >90% of a column is missing, imputing would just be guessing; we drop those columns.

In [ ]:
cols_to_drop = missing[missing > 0.9].index
df = df.drop(columns=cols_to_drop)

In [ ]:
df.info()

**Data Type Conversion**

`income` and `loan_amount` came in as strings because of the `"NA"` / `"Exempt"` markers; we cast with `errors="coerce"` so bad values turn into `NaN`.

We keep race/ethnicity/sex as `category` mostly for memory and so sklearn treats them as discrete in the pipeline.

In [ ]:
# Convert numeric variables
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df["loan_amount"] = pd.to_numeric(df["loan_amount"], errors="coerce")

# Convert categorical variables
cat_cols = ["derived_race", "derived_ethnicity", "derived_sex"]
for col in cat_cols:
    df[col] = df[col].astype("category")

#### 4.	Baseline fairness analysis (pre-model)

Before any model, we look at raw approval rates by race, ethnicity, and sex. Whatever gap shows up here is already in the data, not something the model invented — useful as a reference when we look at model errors later.

In [ ]:
df.groupby("derived_race")["target"].mean().sort_values()

In [ ]:
import matplotlib.pyplot as plt

df.groupby("derived_race")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Race")
plt.show()

**Approval Rate by Race**

White and Asian show higher approval; Black or African American is lower. Joint files sit high, which probably mixes stronger joint income/credit. "Free Form Text Only" is a messy bucket — I would not over-read that level. None of this is model-driven yet.

In [ ]:
df.groupby("derived_ethnicity")["target"].mean()

In [ ]:
df.groupby("derived_ethnicity")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Ethnicity")
plt.show()

**Approval Rate by Ethnicity**

Not Hispanic/Joint are higher; Hispanic and free-text are lower. We have not controlled for income/DTI yet, so this is descriptive only.

In [ ]:
df.groupby("derived_sex")["target"].mean()

In [ ]:
df.groupby("derived_sex")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Sex")
plt.show()

**Approval Rate by Sex**

Male vs female is fairly flat compared to race. Joint is high again. "Sex Not Available" is mostly a missing-code issue.

**Examining Control Variables**

Gaps by race could partly come from who has higher income or safer DTI. We compare income (and later DTI) by race to see how much the raw gap moves once you look at those.

In [ ]:
df.groupby("derived_race").agg({
    "income": "mean",
    "target": "mean"
})

**Comparing Approval Rates and Income Across Racial Groups**

Approval rates run from about 63% (Black or African American) to high 70s (White, Asian). Surprisingly, Black applicants have the *highest* mean income in this sample but the lowest approval — so the gap does not go away if you only look at income. We add DTI buckets and other controls next.

In [ ]:
import numpy as np

def dti_to_numeric(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    
    if "<20%" in x:
        return 15
    elif "20%-<30%" in x:
        return 25
    elif "30%-<36%" in x:
        return 33
    elif "36%-<50%" in x:
        return 43
    elif "50%-60%" in x:
        return 55
    elif ">60%" in x:
        return 65
    
    try:
        return float(x)
    except:
        return np.nan

**DTI (debt-to-income)**

`debt_to_income_ratio` is messy: you get real numbers in some rows and binned strings like `20%-<30%` in others. If you blindly cast to float, you lose a lot of rows.

We map each binned value to a single number in the middle of the range (e.g. `30%-<36%` → 33) so everything is numeric, then bucket into coarse groups (<30%, 30–40%, …) for the tables below.

In [ ]:
df["dti_numeric"] = df["debt_to_income_ratio"].apply(dti_to_numeric)

In [ ]:
def simplify_dti(x):
    if pd.isna(x):
        return "Missing"
    
    if x < 30:
        return "<30%"
    elif x < 40:
        return "30-40%"
    elif x < 50:
        return "40-50%"
    elif x < 60:
        return "50-60%"
    else:
        return ">60%"

In [ ]:
df["dti_group"] = df["dti_numeric"].apply(simplify_dti)

In [ ]:
result = df.groupby(["derived_race", "dti_group"])["target"].mean().unstack()
result["overall_approval_rate"] = df.groupby("derived_race")["target"].mean()
result

**DTI vs approval by race**

Within each DTI bucket, approval still differs by race: everyone drops as DTI gets worse, but White tends to be above the rest at the same bucket. Black or African American and "Two or More Minority Races" are often at the bottom within a bucket, so DTI explains some of the story but not all.

### Modeling  

#### 5.	Feature selection

**Variables in Model A vs B**

We fit two specs on the same outcome: the only change is whether `derived_race` is in the feature list or not.

**Model A — no protected class in X:** `income`, binned **DTI** (`dti_group`), `loan_term`, `loan_type`, `loan_purpose`, `applicant_credit_score_type`, `tract_minority_population_percent`, `loan_to_value_ratio` (CLTV), `property_value`. We keep CLTV + value instead of also throwing in `loan_amount` because CLTV already scales the loan to the house; adding raw amount is redundant. We left **interest_rate** / **rate_spread** and **state_code** out of this main spec — rate fields are set around pricing and can leak post-decision information; state we handle separately in robustness checks.

**Model B — same as A plus** `derived_race`. This is for comparison only (you would not ship a model that conditions on race under normal fair-lending practice). We did not add sex because the raw approval gap by sex was small in our EDA, so the interesting contrast for us was race.

Downstream we compare test accuracy but also **FPR** and **FNR** by race: who gets false approvals vs who gets false denials, not just overall error.


In [ ]:
from sklearn.model_selection import train_test_split

#features without race 
featuresA = [
    "income",
    "dti_group",
    "loan_term",
    "loan_type",
    "loan_purpose",
    "applicant_credit_score_type",
    "tract_minority_population_percent",
    "loan_to_value_ratio",
    "property_value"
    

]
#features with race 
featuresB= featuresA + ["derived_race"]
target = "target"

#### 6. Train-Test Split

Same random 80/20 split and same rows for both models — only the columns differ, so if something moves it is from adding race, not from a different train/test draw.

**Model A (without race)**

In [ ]:
# -----------------------------
# Step 0: define feature types
# -----------------------------
numeric_features_A = [
    "income",
    "tract_minority_population_percent",
    "loan_to_value_ratio",
    "property_value"
]

categorical_features_A = [
    "dti_group",
    "loan_term",
    "loan_type",
    "loan_purpose",
    "applicant_credit_score_type"
]

# -----------------------------
# Step 1: clean dtypes in df
# -----------------------------
for col in numeric_features_A:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in categorical_features_A:
    df[col] = df[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# for Model B
df["derived_race"] = df["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)

# -----------------------------
# Step 2: build X and y
# -----------------------------
X_A = df[featuresA].copy()
y = df[target].astype(int).copy()

# -----------------------------
# Step 3: train/test split
# -----------------------------
X_train_A, X_test_A, y_train, y_test = train_test_split(
    X_A,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# optional check
print(X_train_A.dtypes)
print(X_test_A.dtypes)

n_train = X_train_A.shape[0]
n_test = X_test_A.shape[0]
pct_train = 100 * n_train / (n_train + n_test)
pct_test = 100 * n_test / (n_train + n_test)
print(
    f"Model A → Train: {X_train_A.shape} ({n_train:,} rows, {pct_train:.1f}%), "
    f"Test: {X_test_A.shape} ({n_test:,} rows, {pct_test:.1f}%)"
)

**Model B (with race)**

In [ ]:
# Build X
X_B = df[featuresB].copy()

# Use SAME split indices
X_train_B = X_B.loc[X_train_A.index].copy()
X_test_B  = X_B.loc[X_test_A.index].copy()

# Clean dtypes in X_B using same logic as X_A
for col in numeric_features_A:
    X_train_B[col] = pd.to_numeric(X_train_B[col], errors="coerce")
    X_test_B[col]  = pd.to_numeric(X_test_B[col], errors="coerce")

for col in categorical_features_A:
    X_train_B[col] = X_train_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_B[col] = X_test_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# Clean derived
X_train_B["derived_race"] = X_train_B["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)
X_test_B["derived_race"] = X_test_B["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)


# optional check
print(X_train_B.dtypes)
print(X_test_B.dtypes)

print(
    f"Model B → Train: {X_train_B.shape}, "
    f"Test: {X_test_B.shape}"
)

#### 7.  logistic regression (Model A vs Model B)

**Model A**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


# -----------------------------
# Step 4: preprocessing for Model A
# -----------------------------
numeric_transformer_A = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_A = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_A = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_A, numeric_features_A),
        ("cat", categorical_transformer_A, categorical_features_A)
    ]
)

# -----------------------------
# Step 5: Logistic Regression pipeline
# -----------------------------
model_A = Pipeline(steps=[
    ("preprocessor", preprocessor_A),
    ("classifier", LogisticRegression(max_iter=1000))
])

# -----------------------------
# Step 5.5: guarantee sklearn-safe dtypes (pd.NA → np.nan)
# -----------------------------
for _c in numeric_features_A:
    X_train_A[_c] = pd.to_numeric(X_train_A[_c], errors="coerce").values
    X_test_A[_c]  = pd.to_numeric(X_test_A[_c],  errors="coerce").values
for _c in categorical_features_A:
    X_train_A[_c] = X_train_A[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_A[_c] = X_test_A[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

# -----------------------------
# Step 6: fit model
# -----------------------------
model_A.fit(X_train_A, y_train)

# -----------------------------
# Step 7: predictions
# -----------------------------
y_train_pred_A = model_A.predict(X_train_A)
y_test_pred_A = model_A.predict(X_test_A)

# -----------------------------
# Step 8: evaluation
# -----------------------------
train_acc_A = accuracy_score(y_train, y_train_pred_A)
test_acc_A = accuracy_score(y_test, y_test_pred_A)

print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")

In [ ]:
result_A = X_test_A.copy()
result_A["y_true"] = y_test.values
result_A["y_pred"] = y_test_pred_A
result_A["race"] = df.loc[X_test_A.index, "derived_race"].astype(object).fillna(np.nan)

def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

fairness_A = result_A.groupby("race", dropna=False).apply(compute_metrics)
fairness_A.round(4)

**Model B**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


# -----------------------------
# Step 1: build X for Model B
# -----------------------------
X_B = df[featuresB].copy()

# Use the same split indices as Model A
X_train_B = X_B.loc[X_train_A.index].copy()
X_test_B  = X_B.loc[X_test_A.index].copy()

# -----------------------------
# Step 2: define feature types for Model B
# -----------------------------
numeric_features_B = numeric_features_A.copy()

categorical_features_B = categorical_features_A + ["derived_race"]

# guarantee sklearn-safe dtypes (pd.NA → np.nan)
for col in numeric_features_B:
    X_train_B[col] = pd.to_numeric(X_train_B[col], errors="coerce").values
    X_test_B[col]  = pd.to_numeric(X_test_B[col],  errors="coerce").values
for col in categorical_features_B:
    X_train_B[col] = X_train_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_B[col] = X_test_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# -----------------------------
# Step 3: preprocessing for Model B
# -----------------------------
numeric_transformer_B = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_B = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_B = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_B, numeric_features_B),
        ("cat", categorical_transformer_B, categorical_features_B)
    ]
)

# -----------------------------
# Step 4: Logistic Regression pipeline
# -----------------------------
model_B = Pipeline(steps=[
    ("preprocessor", preprocessor_B),
    ("classifier", LogisticRegression(max_iter=1000))
])

# -----------------------------
# Step 5: fit Model B
# -----------------------------
model_B.fit(X_train_B, y_train)

# -----------------------------
# Step 6: predictions
# -----------------------------
y_train_pred_B = model_B.predict(X_train_B)
y_test_pred_B = model_B.predict(X_test_B)

# -----------------------------
# Step 7: evaluation
# -----------------------------
train_acc_B = accuracy_score(y_train, y_train_pred_B)
test_acc_B = accuracy_score(y_test, y_test_pred_B)

print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_B = model_B.predict(X_test_B)

# Combine results
result_B = X_test_B.copy()
result_B["y_true"] = y_test.values
result_B["y_pred"] = pred_B
result_B["race"] = df.loc[X_test_B.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_B = result_B.groupby("race", dropna=False).apply(compute_metrics)

fairness_B.round(4)

**LR: Model A vs Model B (FPR / FNR / accuracy by race)**

In [ ]:
fairness_A_renamed = fairness_A.rename(columns={
    "FPR": "FPR_A",
    "FNR": "FNR_A",
    "Accuracy": "Accuracy_A",
    "Approval_Rate": "Approval_Rate_A"
}).drop(columns=["Count"])

fairness_B_renamed = fairness_B.rename(columns={
    "FPR": "FPR_B",
    "FNR": "FNR_B",
    "Accuracy": "Accuracy_B",
    "Approval_Rate": "Approval_Rate_B"
}).drop(columns=["Count"])

comparison_lr = fairness_A_renamed.join(fairness_B_renamed)
comparison_lr.round(4)

In [ ]:
print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")
print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")

**Logistic regression — takeaway**

Overall test accuracy is almost the same with or without race, but for Black or African American (and American Indian / Alaska Native) the story gets worse with race in the model: predicted approvals fall and FNR goes up, i.e. more wrong denials. So adding `derived_race` is not a fairness fix here — if anything it tracks the wrong direction in our run.

#### 8. Gradient boost tree (Model A vs Model B)

**Model A**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# Step: Gradient Boosting pipeline (Model A)
# -----------------------------
model_A_gbt = Pipeline(steps=[
    ("preprocessor", preprocessor_A),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# -----------------------------
# Fit model
# -----------------------------
model_A_gbt.fit(X_train_A, y_train)

# -----------------------------
# Predictions
# -----------------------------
y_train_pred_gbt = model_A_gbt.predict(X_train_A)
y_test_pred_gbt = model_A_gbt.predict(X_test_A)

# -----------------------------
# Accuracy
# -----------------------------
train_acc_gbt = accuracy_score(y_train, y_train_pred_gbt)
test_acc_gbt = accuracy_score(y_test, y_test_pred_gbt)

print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")


### 8.1 Test ROC-AUC (no refit)

Quick AUC on the holdout for the two Model A fits we already have — so the number in the writeup matches this cell, not a back-of-the-envelope guess.

In [ ]:
from sklearn.metrics import roc_auc_score

y_test_prob_lr = model_A.predict_proba(X_test_A)[:, 1]
y_test_prob_gbt = model_A_gbt.predict_proba(X_test_A)[:, 1]
test_auc_lr = roc_auc_score(y_test, y_test_prob_lr)
test_auc_gbt = roc_auc_score(y_test, y_test_prob_gbt)

print("========== Test ROC-AUC (Model A, already trained) ==========")
print(f"  Logistic Regression — test AUC: {test_auc_lr:.4f}")
print(f"  Gradient Boosting   — test AUC: {test_auc_gbt:.4f}")

### 8.2 Dropping `tract_minority_population_percent` (retrained)

Same split and settings, but one less column — to see what happens to AUC and to the Black/White **AIR** on predicted approvals if we remove the tract minority share. Fair warning: refitting GBT on the full train slice can take a few minutes.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


TR = "tract_minority_population_percent"
numeric_no = [c for c in numeric_features_A if c != TR]

_num = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
_cat = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
pre_no = ColumnTransformer(
    transformers=[
        ("num", _num, numeric_no),
        ("cat", _cat, categorical_features_A),
    ]
)

Xtr = X_train_A.drop(columns=[TR])
Xte = X_test_A.drop(columns=[TR])
for _c in numeric_no:
    Xtr[_c] = pd.to_numeric(Xtr[_c], errors="coerce").values
    Xte[_c] = pd.to_numeric(Xte[_c], errors="coerce").values
for _c in categorical_features_A:
    Xtr[_c] = Xtr[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
    Xte[_c] = Xte[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

model_lr_no = Pipeline([
    ("preprocessor", pre_no),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model_gbt_no = Pipeline([
    ("preprocessor", pre_no),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
    )),
])

model_lr_no.fit(Xtr, y_train)
model_gbt_no.fit(Xtr, y_train)

race_test = df.loc[X_test_A.index, "derived_race"]

def air_bw(y_pred, race_ser):
    b = (race_ser == "Black or African American").to_numpy()
    w = (race_ser == "White").to_numpy()
    if b.sum() == 0 or w.sum() == 0:
        return float("nan")
    return float(y_pred[b].mean() / y_pred[w].mean())

auc_lr_no = roc_auc_score(y_test, model_lr_no.predict_proba(Xte)[:, 1])
auc_gbt_no = roc_auc_score(y_test, model_gbt_no.predict_proba(Xte)[:, 1])
air_lr_no = air_bw(model_lr_no.predict(Xte), race_test)
air_gbt_no = air_bw(model_gbt_no.predict(Xte), race_test)

print("========== Ablation: TRACT feature removed (retrained) ==========")
print(f"  LR  — test AUC: {auc_lr_no:.4f}  |  pred. AIR Black/White: {air_lr_no:.4f}")
print(f"  GBT — test AUC: {auc_gbt_no:.4f}  |  pred. AIR Black/White: {air_gbt_no:.4f}")
print("========== Baseline: WITH tract (same test set) ==========")
print(f"  LR  — test AUC: {test_auc_lr:.4f}  |  pred. AIR Black/White: {air_bw(model_A.predict(X_test_A), race_test):.4f}")
print(f"  GBT — test AUC: {test_auc_gbt:.4f}  |  pred. AIR Black/White: {air_bw(model_A_gbt.predict(X_test_A), race_test):.4f}")

### 8.3 USDA (`loan_type` = 4)

Code 4 = USDA. The main model does badly on that slice, so we try a GBT fit on USDA train rows only and check AUC on the USDA test piece. The code also sets a simple **manual review** flag for USDA test rows as a cheap operational guardrail.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np
import pandas as pd


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


lt_train = pd.to_numeric(X_train_A["loan_type"], errors="coerce")
lt_test = pd.to_numeric(X_test_A["loan_type"], errors="coerce")
mask_tr = lt_train == 4
mask_te = lt_test == 4

n_tr, n_te = int(mask_tr.sum()), int(mask_te.sum())
print("========== USDA segment (loan_type == 4) ==========")
print(f"  Train count: {n_tr:,}  |  Test count: {n_te:,}")

# Human-in-the-loop flag on the test frame (example policy)
needs_human_review = mask_te.to_numpy()
print(f"  Test rows flagged for human review (USDA): {needs_human_review.sum():,}")

if n_tr < 500 or n_te < 50:
    print("  [Skip sub-model] Not enough USDA rows to fit a stable GBT on this machine run.")
else:
    X_us_tr = X_train_A.loc[mask_tr]
    y_us_tr = y_train.loc[mask_tr]
    X_us_te = X_test_A.loc[mask_te]
    y_us_te = y_test.loc[mask_te]

    for _c in numeric_features_A:
        X_us_tr[_c] = pd.to_numeric(X_us_tr[_c], errors="coerce").values
        X_us_te[_c] = pd.to_numeric(X_us_te[_c], errors="coerce").values
    for _c in categorical_features_A:
        X_us_tr[_c] = X_us_tr[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
        X_us_te[_c] = X_us_te[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

    _num_u = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    _cat_u = Pipeline(steps=[
        ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre_us = ColumnTransformer(
        transformers=[
            ("num", _num_u, numeric_features_A),
            ("cat", _cat_u, categorical_features_A),
        ]
    )
    model_usda_gbt = Pipeline([
        ("preprocessor", pre_us),
        ("classifier", GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
        )),
    ])
    model_usda_gbt.fit(X_us_tr, y_us_tr)

    p_global = model_A_gbt.predict_proba(X_us_te)[:, 1]
    p_sub = model_usda_gbt.predict_proba(X_us_te)[:, 1]
    if y_us_te.nunique() < 2:
        print("  [Skip AUC] USDA test slice has only one class in `target`.")
    else:
        auc_global = roc_auc_score(y_us_te, p_global)
        auc_sub = roc_auc_score(y_us_te, p_sub)
        print(f"  GBT (global model) AUC on USDA test: {auc_global:.4f}")
        print(f"  GBT (USDA-only train) AUC on USDA test: {auc_sub:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_gbt = model_A_gbt.predict(X_test_A)

# ---------- Combine results ----------
result_gbt = X_test_A.copy()
result_gbt["y_true"] = y_test.values
result_gbt["y_pred"] = pred_gbt

result_gbt["race"] = df.loc[X_test_A.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_gbt = result_gbt.groupby("race", dropna=False).apply(compute_metrics)
fairness_gbt.round(4)

**Model B**

In [ ]:
# -----------------------------
# Gradient Boosting pipeline
# -----------------------------
model_B_gbt = Pipeline(steps=[
    ("preprocessor", preprocessor_B),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# -----------------------------
# fit model
# -----------------------------
model_B_gbt.fit(X_train_B, y_train)

# -----------------------------
# predictions
# -----------------------------
y_train_pred_B_gbt = model_B_gbt.predict(X_train_B)
y_test_pred_B_gbt = model_B_gbt.predict(X_test_B)

# -----------------------------
# evaluation
# -----------------------------
train_acc_B_gbt = accuracy_score(y_train, y_train_pred_B_gbt)
test_acc_B_gbt = accuracy_score(y_test, y_test_pred_B_gbt)

print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_B_gbt = model_B_gbt.predict(X_test_B)

# ---------- Combine results ----------
result_B_gbt = X_test_B.copy()
result_B_gbt["y_true"] = y_test.values
result_B_gbt["y_pred"] = pred_B_gbt
result_B_gbt["race"] = df.loc[X_test_B.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_B_gbt = result_B_gbt.groupby("race", dropna=False).apply(compute_metrics)

fairness_B_gbt.round(4)

**GBT: Model A vs Model B (FPR / FNR / accuracy by race)**

In [ ]:
fairness_gbt_A_renamed = fairness_gbt.rename(columns={
    "FPR": "FPR_A_GBT",
    "FNR": "FNR_A_GBT",
    "Accuracy": "Accuracy_A_GBT",
    "Approval_Rate": "Approval_Rate_A_GBT"
}).drop(columns=["Count"], errors="ignore")

fairness_gbt_B_renamed = fairness_B_gbt.rename(columns={
    "FPR": "FPR_B_GBT",
    "FNR": "FNR_B_GBT",
    "Accuracy": "Accuracy_B_GBT",
    "Approval_Rate": "Approval_Rate_B_GBT"
}).drop(columns=["Count"], errors="ignore")

comparison_gbt = fairness_gbt_A_renamed.join(fairness_gbt_B_renamed)
comparison_gbt.round(4)

In [ ]:
print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")
print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

**GBT vs LR**

Gradient boosting wins on AUC/accuracy here — the relationships are not really linear, so LR leaves signal on the table. We use GBT Model A for the rest of the stress tests / security section because it is the stronger workhorse.

#### 9. Comparison Across Modeling Approaches

**Why both LR and GBT?**

LR is easy to stress-test (coefficients line up with the PGD code we do later). GBT is more accurate. If the same fairness issues show up in both, it is less likely to be a quirk of one algorithm.

**Side-by-side tables**

Logistic vs GBT, Model A vs B — numbers in one place below.

In [ ]:
print("Logistic Regression")
display(comparison_lr)

print("Gradient Boosted Tree")
display(comparison_gbt)

In [ ]:
print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")
print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")


print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")
print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

GBT beats LR, but the race gaps in rates / FPR / FNR do not go away. Model B wiggles the metrics without erasing the pattern. That usually means the signal is in the “safe” features too (e.g. tract stuff that lines up with race), not that we picked the wrong algorithm.

#### 10. Interpretation and Insights

**What is left after controls**

Income and DTI help explain some of the raw gap, but not all of it. Dropping race from X does not make the group errors identical; adding race in (Model B) can actually hurt on some groups. The usual story is proxies — tract, product type, etc. — still carrying information that lines up with race.

**What we lead with in the report**

We treat **Model A (no race)** as the “policy” spec — what you would plausibly put in front of a credit decision without protected attributes. **Model B** is a what-if: see what happens to fit and group errors if race is in the model. The slides stress Model A; Model B is there for contrast.

### Lecture 2 — transparency

For a lending-style classifier the usual story is: **income** and **DTI** = can you pay; **CLTV** = how thin the equity cushion is; **loan_term** / **type** / **purpose** = product and horizon. We did not feed raw `loan_amount` into the main spec (CLTV + `property_value` already scale the deal). Anything like SHAP or coef plots in class will point at the same handful of money variables.

### Lecture 3 — fairness

We use **race / ethnicity / sex** to form groups, not as training inputs in Model A. **`tract_minority_population_percent`** is in the model, so it is both a “risk” feature and something that can line up with race — worth watching in the tables. **Income, DTI, CLTV, loan type/purpose** let us see whether a gap is mostly about underwriting risk or something else. Standard outputs: group approval rate, FPR, FNR, and a few cross-tabs if we need them.

### Lecture 4 — robustness

Slice the test set: **loan_type** / **purpose**, **income** / DTI / CLTV buckets, **tract minority %**, **state**. The point is to see whether headline AUC hides a bad segment (USDA is the poster child in our run).

#### 4.1 Train vs test: PSI and KS (numeric features)

Quick drift check on a subsample. **PSI** rules of thumb: &lt;0.10 stable, 0.10–0.25 keep an eye on it, ≥0.25 something moved a lot. **KS** p-value tiny ⇒ the two samples look different. Here both sets are a random split of the same 2024 file, so we expect boring numbers; real deployment would be year-over-year or prod vs training.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Subsample train and test for computational efficiency (full dataset is ~8M rows)
SAMPLE_SIZE = 50_000
rng = np.random.default_rng(42)
train_idx = rng.choice(len(X_train_A), size=min(SAMPLE_SIZE, len(X_train_A)), replace=False)
test_idx  = rng.choice(len(X_test_A),  size=min(SAMPLE_SIZE, len(X_test_A)),  replace=False)

X_train_sample = X_train_A.iloc[train_idx]
X_test_sample  = X_test_A.iloc[test_idx]


def compute_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index using percentile-based bin boundaries."""
    breakpoints = np.unique(np.percentile(expected, np.linspace(0, 100, bins + 1)))
    exp_pcts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    act_pcts = np.histogram(actual,   bins=breakpoints)[0] / len(actual)
    # Replace zeros to avoid log(0) / division by zero
    exp_pcts = np.where(exp_pcts == 0, 1e-6, exp_pcts)
    act_pcts = np.where(act_pcts == 0, 1e-6, act_pcts)
    return float(np.sum((act_pcts - exp_pcts) * np.log(act_pcts / exp_pcts)))


psi_ks_rows = []
for feat in numeric_features_A:
    train_vals = X_train_sample[feat].dropna().values.astype(float)
    test_vals  = X_test_sample[feat].dropna().values.astype(float)

    psi_val            = compute_psi(train_vals, test_vals)
    ks_stat, ks_pval   = stats.ks_2samp(train_vals, test_vals)

    if psi_val < 0.10:
        flag = "STABLE"
    elif psi_val < 0.25:
        flag = "MONITOR"
    else:
        flag = "RETRAIN"

    psi_ks_rows.append({
        "Feature":      feat,
        "PSI":          round(psi_val, 4),
        "PSI_Flag":     flag,
        "KS_Statistic": round(ks_stat, 4),
        "KS_p_value":   round(ks_pval, 6),
    })

psi_df = pd.DataFrame(psi_ks_rows)
print("Distribution Shift Detection — PSI and KS Test (Train vs Test)")
display(psi_df)

**PSI / KS read**

On our run the numerics look stable train vs test and KS is not screaming — which is what you get from a single-year random split. That does *not* mean 2025 data will look the same; for a real system you would still log PSI on live traffic.

#### 4.2 MMD² (joint shift after encoding)

PSI/KS are one feature at a time. **MMD²** on the preprocessed matrix asks whether train and test look different *together* (correlations and all). We use a small subsample and a linear kernel; thresholds below are just a rough scale for the assignment.

In [ ]:
import scipy.sparse as sp

# Subsample to 10 000 rows each side for MMD (quadratic cost in sample size)
MMD_SIZE  = 10_000
rng_mmd   = np.random.default_rng(42)
tr_mmd_idx = rng_mmd.choice(len(X_train_A), size=min(MMD_SIZE, len(X_train_A)), replace=False)
te_mmd_idx = rng_mmd.choice(len(X_test_A),  size=min(MMD_SIZE, len(X_test_A)),  replace=False)

# Transform through the fitted preprocessor (already fitted during modeling)
X_tr_enc = preprocessor_A.transform(X_train_A.iloc[tr_mmd_idx])
X_te_enc = preprocessor_A.transform(X_test_A.iloc[te_mmd_idx])

# Convert sparse matrices to dense arrays if needed
if sp.issparse(X_tr_enc):
    X_tr_enc = X_tr_enc.toarray()
if sp.issparse(X_te_enc):
    X_te_enc = X_te_enc.toarray()

X_tr_enc = X_tr_enc.astype(np.float64)
X_te_enc = X_te_enc.astype(np.float64)


def compute_mmd_linear(X: np.ndarray, Y: np.ndarray) -> float:
    """Unbiased MMD² estimator using linear (dot-product) kernel."""
    n, m = len(X), len(Y)
    XX = np.dot(X, X.T)
    YY = np.dot(Y, Y.T)
    XY = np.dot(X, Y.T)
    mmd2 = (
        (np.sum(XX) - np.trace(XX)) / (n * (n - 1))
        - 2.0 * np.sum(XY) / (n * m)
        + (np.sum(YY) - np.trace(YY)) / (m * (m - 1))
    )
    return float(mmd2)


mmd2 = compute_mmd_linear(X_tr_enc, X_te_enc)
print(f"MMD² (linear kernel, encoded feature space): {mmd2:.6f}")

if mmd2 < 0.01:
    print("Interpretation: LOW shift — train and test distributions are similar; model is stable.")
elif mmd2 < 0.10:
    print("Interpretation: MODERATE shift — some multivariate drift detected; monitor model performance closely.")
else:
    print("Interpretation: HIGH shift — significant distribution change detected; consider retraining.")

**MMD² read**

Our MMD² came out tiny (order 10⁻⁴ in the notebook run), in line with PSI/KS — so for *this* split, train and test look basically the same in the encoded space. Again: that is not a guarantee about next year’s book or out-of-time validation.

#### 4.3 Slices (GBT Model A)

Break the test set by `derived_race` and by `loan_type` and recompute AUC, FPR, etc. We flag a slice if **overall AUC − slice AUC** is above 0.05 or **FPR** is above 0.6 — ad hoc cutoffs, but they catch the USDA mess.

### loan_type 4 (USDA) — the bad slice

On our test run, `loan_type == 4` was small (~8.7k rows) with **AUC ≈ 0.63** and **FPR ≈ 0.90** vs an overall AUC around **0.85**, so the global model is basically misfit for USDA. Plausible reasons: (1) not many USDA rows in train relative to conventional/FHA, (2) different borrower/feature mix, (3) `action_taken` may not line up 1:1 with how USDA guarantees actually work. **Takeaway for the project:** do not use this one model blind on USDA — either a separate model fit on 4s, or route those to manual review. FHA/VA (types 2–3) also dip vs overall but not as hard as USDA.

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

# Align derived_race from the original dataframe using shared index
test_race_series = df.loc[X_test_A.index, "derived_race"]

# GBT Model A predictions on the full test set
y_test_prob_gbt = model_A_gbt.predict_proba(X_test_A)[:, 1]
y_test_pred_gbt = model_A_gbt.predict(X_test_A)
y_test_arr      = y_test.values


def compute_slice_metrics(y_true, y_pred, y_prob, label):
    """Return a dict of performance metrics for a given data slice."""
    if len(y_true) < 100:
        return None
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
    return {
        "Slice":    label,
        "n":        len(y_true),
        "Accuracy": round(acc, 4),
        "AUC":      round(auc, 4),
        "FPR":      round(fpr, 4),
        "FNR":      round(fnr, 4),
    }


# Overall baseline row
overall_row = compute_slice_metrics(y_test_arr, y_test_pred_gbt, y_test_prob_gbt, "OVERALL")
overall_auc = overall_row["AUC"]
slice_rows  = [overall_row]

# Slice by derived_race
race_arr = test_race_series.values
for race_val in sorted(test_race_series.dropna().unique()):
    mask = race_arr == race_val
    row  = compute_slice_metrics(
        y_test_arr[mask], y_test_pred_gbt[mask], y_test_prob_gbt[mask],
        f"Race: {race_val}"
    )
    if row:
        slice_rows.append(row)

# Slice by loan_type
loan_type_arr = X_test_A["loan_type"].values
for lt_val in sorted(X_test_A["loan_type"].dropna().unique()):
    mask = loan_type_arr == lt_val
    row  = compute_slice_metrics(
        y_test_arr[mask], y_test_pred_gbt[mask], y_test_prob_gbt[mask],
        f"loan_type: {lt_val}"
    )
    if row:
        slice_rows.append(row)

# Assemble and flag results
slice_df = pd.DataFrame(slice_rows)
slice_df["AUC_Drop"] = (overall_auc - slice_df["AUC"]).round(4)
slice_df["FLAG"]     = slice_df.apply(
    lambda r: "FLAG" if (r["AUC_Drop"] > 0.05 or r["FPR"] > 0.60) else "", axis=1
)

print("Slice-Based Evaluation — GBT Model A")
display(slice_df.sort_values("AUC_Drop", ascending=False).reset_index(drop=True))

#### 4.4 Toy stress (income −20%, CLTV +20%)

On a subsample of the test set we scale **income** down 20% or **loan_to_value_ratio** up 20% and see how predicted approvals move by race. It is a coarse exercise — not a macro model — but it shows who sits close to the decision boundary.

In [ ]:
import matplotlib.pyplot as plt

# Subsample the test set for stress testing
STRESS_SIZE = 50_000
rng_stress  = np.random.default_rng(42)
stress_idx  = rng_stress.choice(len(X_test_A), size=min(STRESS_SIZE, len(X_test_A)), replace=False)

X_stress_base = X_test_A.iloc[stress_idx].copy()
y_stress      = y_test.iloc[stress_idx].values
race_stress   = df.loc[X_test_A.iloc[stress_idx].index, "derived_race"].values

# Baseline predictions (no shock applied)
baseline_pred = model_A_gbt.predict(X_stress_base)

# Race groups with sufficient representation in the subsample
focal_races = ["White", "Black or African American", "Asian"]

# Define shock scenarios: (feature_name, multiplicative_factor)
scenarios = {
    "Scenario 1: Income -20%":  ("income",                    0.80),
    "Scenario 2: CLTV +20%":   ("loan_to_value_ratio", 1.20),
}

# Populated for the bar chart cell (single source of truth — no hard-coded deltas)
stress_bar_deltas = {}

for scenario_name, (feat, factor) in scenarios.items():
    X_stressed = X_stress_base.copy()
    X_stressed[feat] = X_stressed[feat] * factor

    stressed_pred = model_A_gbt.predict(X_stressed)

    rows = []
    deltas_for_chart = []
    for race_val in focal_races:
        mask = race_stress == race_val
        n    = int(mask.sum())
        if n < 50:
            deltas_for_chart.append(float("nan"))
            continue
        base_rate     = float(baseline_pred[mask].mean())
        stressed_rate = float(stressed_pred[mask].mean())
        delta         = stressed_rate - base_rate
        deltas_for_chart.append(delta)
        rows.append({
            "Race":           race_val,
            "N":              n,
            "Baseline Rate":  round(base_rate,     4),
            "Stressed Rate":  round(stressed_rate, 4),
            "Delta":          round(delta,          4),
        })

    stress_bar_deltas[scenario_name] = deltas_for_chart

    print(f"\n{scenario_name}")
    display(pd.DataFrame(rows))


In [ ]:
# ── Grouped bar chart: Δ approval by race × scenario (values from stress test above) ──
import matplotlib.pyplot as plt
import numpy as np

if "stress_bar_deltas" not in globals() or not stress_bar_deltas:
    raise RuntimeError("Run the stress-test cell above first so stress_bar_deltas is defined.")

scenario_names = list(stress_bar_deltas.keys())
if len(scenario_names) < 2:
    raise RuntimeError("Expected at least two stress scenarios for the grouped bar chart.")

sc1_deltas = stress_bar_deltas[scenario_names[0]]
sc2_deltas = stress_bar_deltas[scenario_names[1]]

races = ["White", "Black or\nAfrican American", "Asian"]
if len(sc1_deltas) != len(races) or len(sc2_deltas) != len(races):
    raise RuntimeError("stress_bar_deltas must have one value per race group.")

x     = np.arange(len(races))
width = 0.35

all_vals = [v for pair in (sc1_deltas, sc2_deltas) for v in pair if np.isfinite(v)]
ymin = min(all_vals + [0.0]) - 0.002
ymax = max(all_vals + [0.0]) + 0.002

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width / 2, sc1_deltas, width, label=scenario_names[0],
               color="#d62728", alpha=0.85)
bars2 = ax.bar(x + width / 2, sc2_deltas, width, label=scenario_names[1],
               color="#1f77b4", alpha=0.85)

ax.set_xlabel("Race Group", fontsize=12)
ax.set_ylabel("Δ Approval Rate", fontsize=12)
ax.set_title("Stress Test — Change in Approval Rate by Race Group", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(races, fontsize=11)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.legend(fontsize=10)
ax.set_ylim(ymin, ymax)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        if not np.isfinite(h):
            continue
        ytxt = h + (0.0004 if h >= 0 else -0.0004)
        va = "bottom" if h >= 0 else "top"
        ax.annotate(
            f"{h:.4f}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3 if h >= 0 else -3),
            textcoords="offset points",
            ha="center", va=va,
            fontsize=9,
            fontweight="bold",
        )

plt.tight_layout()
plt.show()

if len(sc1_deltas) > 1 and np.isfinite(sc1_deltas[1]):
    print(
        "\nIncome shock: largest delta for Black in this draw (",
        f"{sc1_deltas[1]:+.4f} on approval prob), vs other groups —",
        "often interpreted as more mass near the cutoff.",
    )


**Stress read**

**Income −20%:** everyone’s predicted approval falls a bit, but Black moves the most in our subsample — usually read as more people sitting just above the threshold (same idea as a thinner buffer). **CLTV +20%:** the chart shows ~0 change for all groups on our run. That is not a bug: tree models move in steps; 20% on CLTV can leave everyone on the same side of the splits, especially if that feature is not what the trees are splitting on hard. Worth eyeballing `feature_importances_` if you need to back that up in the memo.

In [ ]:
# Week 4 summary table — numbers pulled from the same run as PSI/KS, MMD², slice_df, fairness_gbt, stress cells
from IPython.display import display, Markdown

try:
    _psi_max = float(psi_df["PSI"].max())
    _ks_min_p = float(psi_df["KS_p_value"].min())
    _psi_line = f"All numeric features: max PSI = {_psi_max:.4f} (stable if &lt;0.10); min KS p = {_ks_min_p:.4f}"
except Exception:
    _psi_line = "Re-run Week 4.1 (PSI/KS cell) first."
try:
    _mmd = float(mmd2)
    _mmd_line = f"MMD² = {_mmd:.6f} (ref: &lt;0.01 ≈ low shift in this notebook’s encoded space)"
except Exception:
    _mmd_line = "Re-run MMD cell first."
try:
    _fnr_b = float(fairness_gbt.loc["Black or African American", "FNR"])
    _fnr_w = float(fairness_gbt.loc["White", "FNR"])
    _pp = (_fnr_b - _fnr_w) * 100
    _race_line = f"Black FNR {_fnr_b:.2%} vs White {_fnr_w:.2%} — {_pp:+.2f} pp (test, GBT Model A)"
except Exception:
    _race_line = "Re-run GBT fairness table cell first."
try:
    _usda = slice_df[slice_df["Slice"].astype(str).str.contains("loan_type: 4", na=False)]
    if len(_usda):
        _ur = _usda.iloc[0]
        _usda_line = f"n={int(_ur['n']):,}, AUC={_ur['AUC']}, FPR={_ur['FPR']} (global GBT misfit on this slice)"
    else:
        _usda_line = "USDA (loan_type 4) row not found in slice_df"
except Exception:
    _usda_line = "Re-run slice evaluation cell first."
try:
    if "stress_bar_deltas" in dir() and stress_bar_deltas:
        _k = list(stress_bar_deltas.keys())
        d1 = stress_bar_deltas[_k[0]] if len(_k) else None
        d2 = stress_bar_deltas[_k[1]] if len(_k) > 1 else None
        if d1 is not None and len(d1) >= 2 and np.isfinite(d1[0]) and np.isfinite(d1[1]):
            _inc = f"Black Δ {d1[1] * 100:.2f} pp vs White Δ {d1[0] * 100:.2f} pp on pred. approval — {_k[0]}"
        else:
            _inc = "—"
        if d2 is not None and len(d2) >= 1:
            _cltv = f"Focal groups ~flat ({_k[1]}) on this run; trees may not split hard on CLTV at +20%"
        else:
            _cltv = "—"
    else:
        _inc, _cltv = "(run 4.4 stress cell above first)", "(run 4.4 stress cell above first)"
except Exception:
    _inc, _cltv = "—", "—"

md = f"""---
## Week 4 — Robustness Analysis: Key Insights (this run)

| Dimension | Finding | Severity |
|---|---|---|
| **Distribution Shift (PSI/KS)** | {_psi_line} | Low |
| **Multivariate Drift (MMD²)** | {_mmd_line} | Low |
| **Slice (race / error)** | {_race_line} | High |
| **Slice (USDA / loan_type 4)** | {_usda_line} | High |
| **Stress (Income)** | {_inc} | Medium |
| **Stress (CLTV)** | {_cltv} | Low |

**Bottom line:** For this single-year random split, train–test drift is usually **mild**; the main story is **slices (especially USDA)** and **FNR not equal by race** on the test cut. Do not use headline AUC alone to justify release.

---
"""
display(Markdown(md))


### Lecture 5 — privacy & safety


We run three toy attacks on the LR model: **PGD** style noise on numerics (can you nudge an approval?), **label flips** on a subset of Black train rows (does poisoning move AIR?), and **membership inference** (are train rows more “confident” than test?). Sensitive fields are still race/ethnicity/sex plus things like **tract minority %** that can proxy for them; high-signal money fields can also make individuals easier to single out if someone has side information.

### Before you hand this in

Restart kernel → Run All, then save the `.ipynb` so figures and tables are baked in. Otherwise the TA only sees empty outputs.


#### 5.1 PGD-style evasion (LR Model A)

Simple PGD on the **numeric** columns only: step in the direction of the linear model’s weights, clip back to an **ε** ball in standard-deviation units per feature, 10 steps with **α = ε/10**. Not a perfect threat model for tree models, but it matches the lecture setup and uses the LR coefs we already have.

In [ ]:
# Extract gradient direction from LR coefficients (numeric features come first in the encoded space)
lr_coef    = model_A.named_steps["classifier"].coef_[0]       # shape: (n_encoded_features,)
grad_sign  = np.sign(lr_coef[:len(numeric_features_A)])        # sign gives gradient direction for class 1

# Extract StandardScaler std to convert epsilon from standardized to raw feature space
scaler_stds = (
    model_A.named_steps["preprocessor"]
           .named_transformers_["num"]
           .named_steps["scaler"]
           .scale_
)

# Subsample test set for the attack
ATTACK_SIZE = 5_000
rng_atk     = np.random.default_rng(42)
attack_idx  = rng_atk.choice(len(X_test_A), size=min(ATTACK_SIZE, len(X_test_A)), replace=False)
X_atk_base  = X_test_A.iloc[attack_idx].copy()
race_atk    = df.loc[X_test_A.iloc[attack_idx].index, "derived_race"].values

pgd_rows = []
for eps in [0.1, 0.5, 1.0, 2.0]:
    alpha  = eps / 10           # step size = 1/10 of the total epsilon budget
    X_adv  = X_atk_base.copy()

    for _ in range(10):         # 10 PGD iterations
        for j, feat in enumerate(numeric_features_A):
            raw_alpha = alpha * scaler_stds[j]          # step size converted to raw units
            raw_eps   = eps   * scaler_stds[j]          # budget converted to raw units
            # Gradient ascent step (maximize approval probability)
            X_adv[feat] = X_adv[feat] + raw_alpha * grad_sign[j]
            # Project back into the epsilon-ball around the original input
            X_adv[feat] = X_atk_base[feat] + np.clip(
                X_adv[feat] - X_atk_base[feat], -raw_eps, raw_eps
            )

    base_pred_atk = model_A.predict(X_atk_base)
    adv_pred      = model_A.predict(X_adv)

    white_mask = race_atk == "White"
    black_mask = race_atk == "Black or African American"

    air_base = (
        base_pred_atk[black_mask].mean() / base_pred_atk[white_mask].mean()
        if white_mask.sum() > 0 and black_mask.sum() > 0 else float("nan")
    )
    air_adv = (
        adv_pred[black_mask].mean() / adv_pred[white_mask].mean()
        if white_mask.sum() > 0 and black_mask.sum() > 0 else float("nan")
    )

    pgd_rows.append({
        "epsilon":           eps,
        "Baseline_Approval": round(float(base_pred_atk.mean()), 4),
        "Adv_Approval":      round(float(adv_pred.mean()),       4),
        "Baseline_AIR":      round(float(air_base),              4),
        "Adv_AIR":           round(float(air_adv),               4),
    })

pgd_df = pd.DataFrame(pgd_rows)
print("PGD Evasion Attack Results — LR Model A")
display(pgd_df)

#### 5.2 Label-flip poisoning (Black train rows)

We flip a fraction of **denied** (y=0) **Black** applicants’ train labels to approve, refit LR from scratch, and check test AUC and Black/White AIR. The poison rate is **relative to the pool of Black denials** in the training split (not all Black rows). The curves are the Week-5 deliverable — you can see whether the metric actually moves or if the attack is too small to matter at reasonable rates.

In [ ]:
from sklearn.linear_model import LogisticRegression as LR_fresh
from sklearn.pipeline import Pipeline as Pipeline_fresh
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

rng_poison = np.random.default_rng(42)

# Black rows in training set, and Black *denials* only (true deny→approve flips)
black_train_mask = (
    df.loc[X_train_A.index, "derived_race"] == "Black or African American"
).values
y_train_arr = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.asarray(y_train)
black_denied_idx = np.where(black_train_mask & (y_train_arr == 0))[0]
n_black_denied = len(black_denied_idx)

poison_rates = [0.00, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
poison_rows  = []

# Baseline (clean model already trained)
y_base_pred_proba = model_A.predict_proba(X_test_A)[:, 1]
auc_base          = roc_auc_score(y_test, y_base_pred_proba)

white_test_mask = df.loc[X_test_A.index, "derived_race"].values == "White"
black_test_mask = df.loc[X_test_A.index, "derived_race"].values == "Black or African American"

base_pred_test = model_A.predict(X_test_A)
air_base_val = (
    base_pred_test[black_test_mask].mean() / base_pred_test[white_test_mask].mean()
    if white_test_mask.sum() > 0 and black_test_mask.sum() > 0 else float("nan")
)

poison_rows.append({
    "poison_rate":     0.00,
    "AUC":             round(auc_base,     4),
    "AIR_Black_White": round(air_base_val, 4),
})

for p_rate in poison_rates[1:]:
    y_poisoned = y_train.copy()
    if n_black_denied == 0:
        n_flip = 0
        flip_idx = np.array([], dtype=int)
    else:
        n_flip = int(np.floor(n_black_denied * p_rate))
        n_flip = min(n_flip, n_black_denied)
        if n_flip == 0:
            flip_idx = np.array([], dtype=int)
        else:
            flip_idx = rng_poison.choice(black_denied_idx, size=n_flip, replace=False)
    if n_flip > 0:
        y_poisoned.iloc[flip_idx] = 1  # flip denial -> approval

    poisoned_model = Pipeline_fresh([
        ("preprocessor", preprocessor_A),
        ("classifier",   LR_fresh(max_iter=1000, random_state=42))
    ])
    poisoned_model.fit(X_train_A, y_poisoned)

    y_adv_proba = poisoned_model.predict_proba(X_test_A)[:, 1]
    auc_adv     = roc_auc_score(y_test, y_adv_proba)
    adv_pred    = poisoned_model.predict(X_test_A)
    air_adv_val = (
        adv_pred[black_test_mask].mean() / adv_pred[white_test_mask].mean()
        if white_test_mask.sum() > 0 and black_test_mask.sum() > 0 else float("nan")
    )

    poison_rows.append({
        "poison_rate":     p_rate,
        "AUC":             round(auc_adv,     4),
        "AIR_Black_White": round(air_adv_val, 4),
    })

poison_df = pd.DataFrame(poison_rows)
print(
    f"Label-Flip Poisoning — LR Model A | Black denials in train: {n_black_denied:,} | "
    "flips are deny→approve only"
)
display(poison_df)

# Degradation curves: test AUC and AIR vs. poison rate (Lecture 5 deliverable)
pr_pct = poison_df["poison_rate"] * 100
fig, (ax_auc, ax_air) = plt.subplots(1, 2, figsize=(10, 4))
ax_auc.plot(pr_pct, poison_df["AUC"], "o-", color="#1f77b4", linewidth=2, markersize=6)
ax_auc.set_xlabel("Poison rate (% of Black train denials flipped to approve)")
ax_auc.set_ylabel("Test AUC (clean holdout)")
ax_auc.set_title("Performance degradation under poisoning")
ax_auc.grid(True, alpha=0.3)
ax_air.plot(pr_pct, poison_df["AIR_Black_White"], "s-", color="#ff7f0e", linewidth=2, markersize=6)
ax_air.axhline(0.80, color="red", linestyle="--", alpha=0.6, label="80% rule (illustrative)")
ax_air.set_xlabel("Poison rate (% of Black train denials flipped to approve)")
ax_air.set_ylabel("AIR (Black / White pred. approval rate)")
ax_air.set_title("Fairness metric under poisoning")
ax_air.legend(loc="best", fontsize=8)
ax_air.grid(True, alpha=0.3)
fig.suptitle("Label-Flip Poisoning: Degradation Curves", fontweight="bold", fontsize=12)
fig.tight_layout()
plt.show()

**Poisoning read**

The attack only flips **denied** Black training rows (true deny→approve). The horizontal axis is a fraction of *that* denial pool, not all Black rows. AIR may still move slowly if the absolute number of flips is small relative to the full training set, or if LR is stiff. A real bank would still watch for label tampering; the homework point is: *measured* damage can look small until the attack is large in absolute rows.

#### 5.3 Membership inference (MI)

Compare `predict_proba` on a sample of **train** rows vs a sample of **test** rows; if the model is memorizing, members can get systematically higher scores and a classifier could separate them (AUC &gt; 0.5). We print MI AUC, the mean score gap, and a ROC + overlaid histograms so it matches the lecture “ROC + split histogram” ask.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Sample members (from training set) and non-members (from test set)
MI_SIZE = 5_000
rng_mi  = np.random.default_rng(42)

member_idx     = rng_mi.choice(len(X_train_A), size=min(MI_SIZE, len(X_train_A)), replace=False)
nonmember_idx  = rng_mi.choice(len(X_test_A),  size=min(MI_SIZE, len(X_test_A)),  replace=False)

X_member    = X_train_A.iloc[member_idx]
X_nonmember = X_test_A.iloc[nonmember_idx]

# Get model confidence scores (probability of class 1)
conf_member    = model_A.predict_proba(X_member)[:, 1]
conf_nonmember = model_A.predict_proba(X_nonmember)[:, 1]

# Use confidence score as membership signal: higher confidence -> more likely member
mi_labels  = np.concatenate([np.ones(len(conf_member)), np.zeros(len(conf_nonmember))])
mi_scores  = np.concatenate([conf_member, conf_nonmember])
mi_auc     = roc_auc_score(mi_labels, mi_scores)

print(f"Membership Inference AUC: {mi_auc:.4f}")
if mi_auc > 0.70:
    print("Risk Level: HIGH — significant memorization detected; apply differential privacy")
elif mi_auc > 0.60:
    print("Risk Level: MODERATE — some memorization; monitor and regularize")
else:
    print("Risk Level: LOW — model generalizes well; membership largely indistinguishable")

print(f"\nMean confidence (members):     {conf_member.mean():.4f}")
print(f"Mean confidence (non-members): {conf_nonmember.mean():.4f}")
gap = float(conf_member.mean() - conf_nonmember.mean())
print(f"Confidence gap:                {gap:.4f}")

# ROC for membership attack + score overlap (Lecture 5: ROC + confidence gap)
fpr_mi, tpr_mi, _ = roc_curve(mi_labels, mi_scores)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr_mi, tpr_mi, color="#2ca02c", linewidth=2, label=f"MI attack (AUC = {mi_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC = 0.5)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("Membership inference ROC curve")
axes[0].legend(loc="lower right")
axes[0].set_aspect("equal", adjustable="box")
axes[0].grid(True, alpha=0.3)
axes[1].hist(conf_nonmember, bins=40, alpha=0.65, label="Non-members (holdout test)", color="#1f77b4", density=True)
axes[1].hist(conf_member, bins=40, alpha=0.65, label="Members (train sample)", color="#d62728", density=True)
axes[1].set_xlabel("Model confidence: P(approve | x)")
axes[1].set_ylabel("Density")
axes[1].set_title("Confidence gap: training vs. holdout score distributions")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.suptitle(
    f"Membership inference: ROC + score distributions (mean confidence gap = {gap:+.4f})",
    fontweight="bold",
    fontsize=11,
)
fig.tight_layout()
plt.show()

# --- Exported for downstream summary table (Week 5 key insights) ---
MI_AUC = float(mi_auc)
MI_CONFIDENCE_GAP = float(gap)
if MI_AUC > 0.70:
    MI_RISK_LEVEL = "HIGH"
    MI_RISK_LABEL = "High"
elif MI_AUC > 0.60:
    MI_RISK_LEVEL = "MEDIUM"
    MI_RISK_LABEL = "Medium"
else:
    MI_RISK_LEVEL = "LOW"
    MI_RISK_LABEL = "Low"


In [ ]:
# ── Week 5 — Privacy & Safety: Key Insights (numbers from this run) ──
from IPython.display import display, Markdown

def _need(name):
    raise RuntimeError(f"Run the {name} cells above first (PGD, poisoning, membership inference).")

if "pgd_df" not in globals():
    _need("PGD")
if "poison_df" not in globals():
    _need("poisoning")
if "MI_AUC" not in globals():
    _need("membership inference")

pgd_row = pgd_df.loc[pgd_df["epsilon"] == 2.0].iloc[0]
b_app = float(pgd_row["Baseline_Approval"])
a_app = float(pgd_row["Adv_Approval"])
delta_pp = round((a_app - b_app) * 100, 2)
air_b = float(pgd_row["Baseline_AIR"])
air_a = float(pgd_row["Adv_AIR"])

air0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AIR_Black_White"].iloc[0])
air_p = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AIR_Black_White"].iloc[0])
auc0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AUC"].iloc[0])
auc_p = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AUC"].iloc[0])

mi_line = (
    f"MI AUC = **{MI_AUC:.4f}**; mean confidence gap = **{MI_CONFIDENCE_GAP:+.4f}** "
    f"({MI_RISK_LEVEL} per the bands in the MI markdown cell)"
)

md = f"""
---
## Week 5 — Privacy & Safety: Key Insights

| Attack Vector | Key Result | Severity |
|---|---|---|
| **PGD Evasion (ε = 2.0)** | Approval rate **{b_app:.3f} → {a_app:.3f}** (Δ **+{delta_pp:.2f} pp**); AIR **{air_b:.4f} → {air_a:.4f}** | Medium |
| **Label-Flip Poisoning (20%)** | Test AUC **{auc0:.4f} → {auc_p:.4f}**; AIR **{air0:.4f} → {air_p:.4f}** | Medium |
| **Membership Inference** | {mi_line} | {MI_RISK_LABEL} |

**PGD:** LR is easy to nudge; at ε=2.0 we see about **+{delta_pp:.2f} pp** on mean predicted approval in the attack subsample. AIR can *move* in a way that looks "fairer" if both groups shift together — that is not proof the model is safe, just that this attack is symmetric in practice.

**Poisoning:** at 20% flips the curves above show how much AUC/AIR actually move; LR can shrug off small flips for the reasons in the text cell.

**MI:** AUC **{MI_AUC:.4f}** with gap **{MI_CONFIDENCE_GAP:+.4f}** — {MI_RISK_LEVEL} per our cutoffs. HMDA is still sensitive data; low MI is not a pass to show raw scores in public.

### Before turn-in: restart, run all, save the notebook.

---
"""
display(Markdown(md))


### Five capstone questions (short answers below)

The next cells are our slide-ready blurbs: objective, what can go wrong, subgroup errors, residual risks, and whether we would ship this. Numbers pull from the same tables as the rest of the notebook when you have run everything.

In [ ]:
print("=" * 70)
print("Q1: What is the optimization objective of this model?")
print("=" * 70)
print("""
We minimize log loss on a binary approved/denied target from HMDA `action_taken`,
80/20 stratified split, 2024 LAR. Model A (what we call the "policy" spec) does not
put race/ethnicity/sex in the feature vector — for the assignment the interesting
numeric + tract fields are: income, CLTV, tract minority %, property value, plus the
categorical loan fields in `featuresA`.

We fit both logistic regression and a gradient boosting classifier on the same split;
GBT is higher AUC (roughly 0.85 vs 0.80) but the fairness picture does not flip when
we switch models, so the takeaways are mostly about the data, not the algorithm label.
""")

In [ ]:
print("=" * 70)
print("Q2: What are the known failure modes of this model?")
print("=" * 70)

_pgd = pgd_df.loc[pgd_df["epsilon"] == 2.0].iloc[0]
_pgd_dpp = (float(_pgd["Adv_Approval"]) - float(_pgd["Baseline_Approval"])) * 100

_air0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AIR_Black_White"].iloc[0])
_air20 = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AIR_Black_White"].iloc[0])
_auc0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AUC"].iloc[0])
_auc20 = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AUC"].iloc[0])
_auc_drop = _auc0 - _auc20
_air_drop = _air0 - _air20

_mi_sev = "High" if mi_auc > 0.70 else ("Medium" if mi_auc > 0.60 else "Low")
_gap = MI_CONFIDENCE_GAP if "MI_CONFIDENCE_GAP" in globals() else float("nan")

# Slice / subgroup evidence from this run (GBT test — keeps Q2 aligned with tables above)
try:
    _fnr_b = float(fairness_gbt.loc["Black or African American", "FNR"])
    _fnr_w = float(fairness_gbt.loc["White", "FNR"])
    _fpr_j = float(fairness_gbt.loc["Joint", "FPR"])
    _pp_fnr = (_fnr_b - _fnr_w) * 100
    _blk = slice_df[slice_df["Slice"].str.contains("Black or African American", na=False)]
    if len(_blk) and "AUC_Drop" in _blk.columns:
        _ad_pp = float(_blk["AUC_Drop"].iloc[0]) * 100
        _auc_b = float(_blk["AUC"].iloc[0])
    else:
        _ad_pp, _auc_b = float("nan"), float("nan")
    _slice_evd = (
        f"FNR: Black ({_fnr_b:.2%}) vs White ({_fnr_w:.2%}) — {_pp_fnr:+.2f} pp; Joint FPR = {_fpr_j:.2%}. "
        f"Black race slice AUC = {_auc_b:.4f}; AUC drop vs overall ≈ {_ad_pp:.1f} pp (ad hoc FLAG if drop > 5 pp or FPR high)."
    )
except Exception:
    _slice_evd = "Re-run Week 4 slice + GBT fairness cells so `fairness_gbt` and `slice_df` exist."

failure_modes_df = pd.DataFrame([
    {
        "Failure Mode":   "Proxy Discrimination",
        "Evidence":       "tract_minority_population_percent retained as feature despite being a geographic proxy for race; pre-model AIR < 0.80 for Black applicants. Removing the feature reduces AUC but is expected to improve AIR — a tradeoff acknowledged as a deployment risk.",
        "Severity":       "High",
        "Mitigation":     "Pre-deployment: ablation study to quantify AUC vs AIR tradeoff upon removing tract_minority_population_percent. Interim: flag as high-risk in model card; apply fairness-aware regularization (e.g., adversarial debiasing).",
    },
    {
        "Failure Mode":   "Distribution Shift (Covariate Drift)",
        "Evidence":       "PSI and KS tests confirm all numeric features STABLE (PSI < 0.10); "
                         "no covariate drift detected between train/test, but production drift "
                         "remains a forward risk if economic conditions change",
        "Severity":       "Medium",
        "Mitigation":     "Schedule periodic retraining; implement automated PSI monitoring pipeline",
    },
    {
        "Failure Mode":   "Adversarial Evasion (PGD)",
        "Evidence":       (
            f"PGD at ε=2.0 (LR): approval {float(_pgd['Baseline_Approval']):.3f}→{float(_pgd['Adv_Approval']):.3f} "
            f"(+{_pgd_dpp:.2f} pp); AIR {float(_pgd['Baseline_AIR']):.4f}→{float(_pgd['Adv_AIR']):.4f} — "
            f"gradient-based evasion is material even when AIR moves favorably"
        ),
        "Severity":       "Medium",
        "Mitigation":     "Adversarial training; input range validation; feature value clamping",
    },
    {
        "Failure Mode":   "Label-Flip Data Poisoning",
        "Evidence":       (
            f"Targeted flips on Black train *denials* (up to 20% of that pool): test AUC {_auc0:.4f}→{_auc20:.4f} (Δ={_auc_drop:+.4f}); "
            f"AIR {_air0:.4f}→{_air20:.4f} (Δ={_air_drop:+.4f})"
        ),
        "Severity":       "High" if (_auc_drop > 0.02 or abs(_air_drop) > 0.05) else "Medium",
        "Mitigation":     "RONI filter; cryptographic data integrity checks; data provenance logging",
    },
    {
        "Failure Mode":   "Membership Inference (Privacy Leakage)",
        "Evidence":       (
            f"MI AUC = {mi_auc:.3f} (see MI section: {_mi_sev.lower()} risk band); "
            f"mean confidence gap = {_gap:+.4f}"
        ),
        "Severity":       _mi_sev,
        "Mitigation":     "Differential privacy (DP-SGD); prediction confidence clipping; API rate limiting",
    },
    {
        "Failure Mode":   "Slice-Level Performance Disparity",
        "Evidence":       _slice_evd,
        "Severity":       "High",
        "Mitigation":     "Equalized odds post-processing; per-subgroup threshold calibration",
    },
])

display(failure_modes_df)


In [ ]:
print("=" * 70)
print("Q3: How does subgroup error vary across protected groups?")
print("=" * 70)

from sklearn.metrics import confusion_matrix

# Pull race slices from the slice evaluation results
race_groups   = ["White", "Black or African American", "Asian", "Native Hawaiian or Other Pacific Islander", "Joint"]
q3_rows       = []

for race_val in race_groups:
    mask = df.loc[X_test_A.index, "derived_race"].values == race_val
    if mask.sum() < 50:
        continue
    y_true_slice = y_test.values[mask]
    y_pred_slice = model_A_gbt.predict(X_test_A.iloc[np.where(mask)[0]])

    tn, fp, fn, tp = confusion_matrix(
        y_true_slice, y_pred_slice, labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    q3_rows.append({
        "Race Group": race_val,
        "N":          int(mask.sum()),
        "FPR":        round(fpr, 4),
        "FNR":        round(fnr, 4),
    })

q3_df = pd.DataFrame(q3_rows)
print("Subgroup Error Rates — GBT Model A")
display(q3_df)
print("\nIn this test split Black FNR is higher than White — same direction as the fairness section above.")

In [ ]:
print("=" * 70)
print("Q4: What residual risks remain after analysis?")
print("=" * 70)

mi_risk_level = "High" if mi_auc > 0.70 else ("Medium" if mi_auc > 0.60 else "Low")

try:
    _usda_row = slice_df[slice_df["Slice"].astype(str).str.contains("loan_type: 4", na=False)]
    if len(_usda_row):
        _u = _usda_row.iloc[0]
        _usda_status = f"Open — sub-model or exclusion (this run: n={int(_u['n'])}, AUC={_u['AUC']}, FPR={_u['FPR']})"
    else:
        _usda_status = "Open — sub-model or exclusion required"
except Exception:
    _usda_status = "Open — sub-model or exclusion required (re-run Week 4 slice cell)"

residual_risks = pd.DataFrame([
    {"Risk": "Proxy discrimination via tract_minority_population_percent",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — ablation study pending"},
    {"Risk": "No automated PSI monitoring in production",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — pipeline not yet built"},
    {"Risk": "Membership inference privacy risk",
     "Likelihood": mi_risk_level, "Impact": "High" if mi_auc > 0.60 else "Medium",
     "Status": f"Open — MI AUC = {mi_auc:.3f}; DP not yet applied"},
    {"Risk": "USDA loan slice failure (global GBT misfit on loan_type 4)",
     "Likelihood": "High", "Impact": "Medium",
     "Status": _usda_status},
    {"Risk": "PGD evasion attack surface",
     "Likelihood": "Medium", "Impact": "Medium",
     "Status": "Open — adversarial training not implemented"},
    {"Risk": "Equalized Odds violation (Black/White FNR gap)",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — post-processing not applied"},
])

display(residual_risks)
print(f"\nTotal open risks: {len(residual_risks)}")
print(f"High likelihood x High impact: {len(residual_risks[(residual_risks.Likelihood=='High') & (residual_risks.Impact=='High')])}")


#### Q5: Would we deploy this as-is?

**Short answer: no** — for a class project we are happy with the *analysis*, but a real lender would not ship this without fixing at least: proxy risk from **tract minority %** (or proving it is OK), the **USDA** slice, **FNR** gaps by race, and a proper **monitoring** plan; privacy attacks above are more “what-if” but still a reminder to lock down APIs. GBT is accurate on average; the problems are in the corners.


## Week 6 — Conclusion: tying the DNSC 6330 arc to this HMDA build

The course basically walks from **what the model optimizes** and **how you explain it** (Lectures 1–2), through **fairness metrics that match a decision** (Lecture 3), into **whether the world stays like training** and **where it breaks in slices** (Lecture 4), and finally **what an adversary or a curious client can do** (Lecture 5). Week 6 is the boring-sounding part everyone skips in slides—**governance**—except it is where all of that either becomes an actionable risk register or stays a pile of plots.

For our notebook, the through-line is blunt: **a strong GBT AUC on a stratified 80/20 split is not a release argument.** It answers a narrow statistical question. It does not answer whether a deny hurts some communities more (FNR), whether a tract-level field is doing the work of race (proxy risk), whether `loan_type == 4` (**USDA**) is a pocket where the global model is wrong, or whether the scoring API leaks membership when someone can query it repeatedly. Those are different questions, and Lectures 3–5 give the vocabulary and the sanity checks for each.

### Lecture-by-lecture hook to *this* artifact

**Lecture 1 (framing / model risk).** We treated approval modeling as a supervised problem on HMDA `action_taken`, but the responsible-ML framing is really **model risk + consumer impact**: the loss we minimize (Q1) is not the same object as fair lending compliance, robustness under shift, or privacy. Keeping that separation explicit is why Q1 is not just "we used sklearn."

**Lecture 2 (transparency).** PDP/ICE, LIME, SHAP, counterfactuals are about **local and global story-telling**. Our main fairness story here is not only "what the model looked at" but whether **errors line up with protected groups** once you fix the threshold narrative (we report FPR/FNR by `derived_race` rather than hiding behind accuracy). Transparency tools in production sit next to **monitoring**: if slice metrics move, explanations that looked fine in development can lie by omission.

**Lecture 3 (fairness).** We leaned hard on **AIR**, **subgroup FPR/FNR**, and the **Model A vs B** contrast (race in vs out of `featuresA`). That matches the Lecture 3 emphasis on **adverse impact** and **equalized-odds-style** thinking: the interesting finding is that **Black FNR &gt; White FNR** on the GBT test split for Model A, i.e. conditional error rates are not aligned even when race is not a direct input—classic proxy / correlation structure. A full fair-lending review would also bring in **calibration**, **SMD**, and formal tests; we did not build those tables here, but they land in the same bucket as "things the validator asks after seeing your EO gaps."

**Lecture 4 (bias → robustness).** **Robustness is not the same as IID generalization.** PSI/KS/MMD$^2$ on our train vs test split mostly said "no obvious covariate explosion in this single snapshot," which is weak comfort: production drift is **forward-looking**. The more actionable Lecture 4 output in our run is **slice evaluation**: the **USDA** slice and race slices in `slice_df` are where the average AUC masks failure. Stress-style perturbations (income/CLTV) are another way of saying the decision boundary should not be brittle to plausible edits.

**Lecture 5 (privacy & safety).** The PGD toy on logistic regression, the label-flip poisoning curve, and membership-inference ROC/confidence-gap material are deliberately **small-scale**: they show **attack surfaces** (evasion, data integrity, query access) rather than claiming nation-state realism. Still, they belong in Q2/Q4 because a governance discussion that only talks about AUROC is incomplete.

### The five questions as a mini–model card

Think of Q1–Q5 as the shortest version of what a model card or an internal committee memo would demand:

| Gate | What we documented | Why it maps to class |
|---|---|---|
| **Q1** | Log loss on approve/deny; **Model A** keeps race/ethnicity/sex out of `featuresA` | Objective + "main spec" vs counterfactual spec |
| **Q2** | `failure_modes_df`: proxy, drift, PGD, poisoning, MI, slice disparity | Lecture 4–5 failure taxonomy with evidence strings |
| **Q3** | Per-race FPR/FNR on **GBT Model A** | Lecture 3 subgroup error, EO-adjacent |
| **Q4** | `residual_risks` (USDA, monitoring gaps, MI, EO gap, PGD) | Open issues with likelihood/impact |
| **Q5** | **No** ship-as-is | Honest deployment stance |

If you squint, that is the **SR 11-7 style** rhythm without the PDF boilerplate: identify risks, tie them to evidence, escalate what is still open.

### What would actually change with more time (and a real bank)

A classroom notebook stops at tables; a production package adds **owners, due dates, and monitoring hooks**. Concretely for *our* findings: (1) **ablate** `tract_minority_population_percent` and document the AUC–fairness trade explicitly; (2) **either** exclude USDA (`loan_type == 4`) from the global score **or** fit a slice-specific checker with its own thresholds; (3) if EO gaps persist, **thresholding or post-processing** has to be on the table, not hand-waved; (4) drift tooling (PSI jobs, slice dashboards) has to exist **before** go-live, not after someone notices AUC dropped; (5) API design for scores should assume **MI-style** adversaries unless DP or strong access controls are in place.

### Closing reflection

HMDA is almost *too convenient* for teaching: public labels, huge $N$, and a story that already matters for fair lending. The downside is forgetting what the target is—not "the bank's proprietary decision," but a **regulatory reporting code** that compresses a messy process into 1/2/3. We used that target because the assignment needs a clean $y$, not because we are endorsing it as a perfect causal notion of "should have approved." What we *are* endorsing is the workflow: **same split for A vs B**, **report subgroup errors**, **hunt slices**, **run cheap attacks**, then **say no** when the risk stack is still open (Q5).

---